# 01 — Adquisición de datos

**Tesis:** Impacto de variables macroeconómicas globales y financieras en la valoración bursátil del sector minería de cobre en Chile.

Este notebook **descarga** lo automatizable (Yahoo + FRED) y **lista** lo que debes bajar a mano (BCCh, Cochilco, CMF) a `data/raw/`.

> ⚠️ No se inventan datos. Verifica la cobertura temporal real de cada serie.

In [ ]:
import sys, os
from pathlib import Path
# Permite importar el paquete src estando dentro de notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config as C
from src import data_acquisition as da
from src import transformations as T

print('Período:', C.FECHA_INICIO, '->', C.FECHA_FIN)
print('Universo activo:', list(C.UNIVERSO_ACTIVO.keys()))

## 1. Descarga automatizada (Yahoo + FRED)
Requiere conexión a internet y `yfinance`, `pandas-datareader` instalados.

In [ ]:
bundle = da.adquirir_todo(guardar=True)
bundle.keys()

In [ ]:
# Inspección rápida
precios = bundle['precios']
display(precios.tail())
display(bundle['macro_global'].tail())

## 2. Carga manual de series del Banco Central de Chile
Descarga desde la **Base de Datos Estadísticos del BCCh** (https://si3.bcentral.cl) y
**Cochilco** (https://www.cochilco.cl) los archivos y guárdalos en `data/raw/` con los
nombres definidos en `config.MACRO_LOCAL_BCCH`.

Ajusta `sep=';'` y `decimal=','` según el formato exportado (el BCCh suele usar esos).

In [ ]:
# Ejemplo de carga (descomentar cuando tengas los archivos):
# tpm    = da.load_local_csv('bcch_tpm.csv',    sep=';', decimal=',')
# imacec = da.load_local_csv('bcch_imacec.csv', sep=';', decimal=',')
# embi   = da.load_local_csv('bcch_embi.csv',   sep=';', decimal=',')
# ipc    = da.load_local_csv('bcch_ipc.csv',    sep=';', decimal=',')
print('Pendiente: ', [v[0] for v in C.MACRO_LOCAL_BCCH.values() if not str(v[0]).startswith("^")])

## 3. Construcción de retornos y cartera del sector

In [ ]:
# Retornos diarios -> remuestreo a fin de mes -> retorno mensual
precios_m = T.a_fin_de_mes(precios, metodo='last')
retornos_m = T.retorno_log(precios_m)

# Cartera del sector (baseline serie de tiempo)
cartera_eq = T.cartera_equiponderada(retornos_m)
cartera_eq.name = 'cartera_sector_eq'
display(retornos_m.tail())
cartera_eq.plot(title='Retorno mensual cartera equiponderada del sector cobre');

## 4. Guardar dataset intermedio
El panel final se arma en el notebook 02 tras los tests de estacionariedad.

In [ ]:
retornos_m.to_parquet(C.DATA_INTERIM / 'retornos_mensuales.parquet')
cartera_eq.to_frame().to_parquet(C.DATA_INTERIM / 'cartera_sector.parquet')
print('Guardado en', C.DATA_INTERIM)